In [ ]:
import os
from dotenv import load_dotenv
import gradio as gr

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS       # <--- vector store
from langchain_huggingface import HuggingFaceEmbeddings  # <--- to enable embedding models
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Load API key safely from .env
load_dotenv()

## 1. Documents for the demo

In [ ]:
docs = [
    Document(page_content="LangChain is a framework for building LLM applications."),
    Document(page_content="RAG means Retrieval-Augmented Generation."),
    Document(page_content="BM25 is a keyword-based retrieval algorithm."),
    Document(page_content="Vector databases are useful for semantic search."),
    Document(page_content="Paris is the capital of France."),
    Document(page_content="Denis Molin is a great guy."),
]

## 2. Retriever / vector store

In [ ]:
embeddings  = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)

## 3. LLM

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 4. RAG function

In [ ]:
SYSTEM_PROMPT = """
You are a helpful assistant answering only from the provided context.

If the answer is not in the context, say:
"I don't know based on the provided documents."

Do not invent information.
"""

def build_rag_answer(vectorstore, llm, k=4):
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})

    prompt = ChatPromptTemplate.from_template("""
{system_prompt}

Context:
{context}

Question:
{question}

Answer:
""")

    def rag_answer(question: str) -> str:
        docs = retriever.invoke(question)
        context = "\n\n".join(doc.page_content for doc in docs)

        messages = prompt.format_messages(
            system_prompt=SYSTEM_PROMPT,
            context=context,
            question=question,
        )

        response = llm.invoke(messages)

        if hasattr(response, "content"):
            return str(response.content)
        return str(response)

    return rag_answer

## 5. Chat interface

In [ ]:
from rag_ui import launch_rag_ui
from rag_vector import build_rag_answer

# Example:
# vectorstore = FAISS.from_documents(chunks, embedding_model)
# llm = ChatOpenAI(model="gpt-4o-mini")  # or your Mistral / LiteLLM wrapper

rag_answer = build_rag_answer(vectorstore=vectorstore, llm=llm, k=4)

def chat(message, history):
    history = history or []
    answer = rag_answer(message)

    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": str(answer)},
    ]
    return history, ""

demo = launch_rag_ui(
    chat_fn=chat,
    title="Simple RAG Demo (Vector Embeddings)",
    port=7860,
)

In [ ]:
from rag_ui import launch_rag_ui

def chat(message, history):
    history = history or []

    answer = rag_answer(message)
    answer = str(answer)

    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer},
    ]

    return history, ""

launch_rag_ui(chat, title="Simple RAG Demo (LangChain + BM25)")